# **RAG Pipeline — Harry Potter Books**

End-to-end pipeline for a Retrieval-Augmented Generation chatbot over the 7 Harry Potter books:

1. Parse the PDF into per-page text
2. Structure pages by book / chapter and save a clean Markdown file
3. Clean and preprocess the text
4. Chunk the text — **page-level chunking** (one chunk = one physical page)
5. Embed the chunks with a multilingual E5 embedding model
6. Store the embeddings in a local FAISS index and save the corresponding page metadata
7. Run retrieval sanity checks, generation, and evaluation

This notebook builds the local knowledge base that `rag_api.py` (the FastAPI app) queries at runtime.

**The embedding model and metadata structure used here must match what `rag_api.py` expects.** The same embedding model must be used when creating the FAISS index and when embedding incoming user queries. For the E5 model, stored pages use the `"passage: "` prefix, while user queries use the `"query: "` prefix.

The FAISS index and metadata are stored locally, so the chatbot does not require a Qdrant Cloud connection at runtime.


## 0. Setup


In [1]:
import os
import re
import json
from pathlib import Path

from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # reads the .env file in this folder

PDF_PATH = os.getenv("PDF_PATH", "Harry Potter.pdf")
MD_PATH = os.getenv("MD_PATH", "harry_potter.md")

# IMPORTANT: this must be the exact same model (and same env var name) that
# rag_api.py uses at query time, or retrieval will silently return garbage —
# either a dimension mismatch (the request will error) or, worse, a same-size
# but semantically different vector space (the request will "succeed" with
# irrelevant results). intfloat/multilingual-e5-large is an E5-family model:
# it expects a "query: " / "passage: " prefix on the text it embeds.
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "intfloat/multilingual-e5-large")

MIN_PAGE_CHARS = int(
    os.getenv("MIN_PAGE_CHARS", 30)
)  # skip near-empty pages (blanks, dividers)

## 1. Read the document & understand its structure

The PDF is _"Harry Potter: The Complete Collection"_ — all 7 books in one file (3,623 pages).

What a quick inventory shows:

- Text is embedded and cleanly extractable (no OCR needed)
- We extract with `PyMuPDF` (`fitz`) — fast, self-contained (no external binary like poppler's `pdftotext` to install), and it hands us text already split per physical page
- Each chapter's first page has an optional stylized drop-cap letter, then `CHAPTER <NUMBER-WORD>` on its own line, then the title in caps — e.g. `"M\nCHAPTER ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley..."` (the `M` + `r.` need to be re-joined into `Mr.`)
- Two chapters ("Snape's Worst Memory", "The Second War Begins") open with an in-world document instead of a paragraph, so they have **no** drop-cap — the header-stripping logic below has to handle both cases
- Chapter numbering restarts at `ONE` at the start of each of the 7 books — that reset is what we use to detect book boundaries
- Book 7 has a final `EPILOGUE` section after chapter 36, detected the same way as a chapter header
- Table-of-contents pages (`CONTENTS` at the top) appear once per book and are filtered out


## 2. Parse the PDF into pages


In [2]:
import fitz  # PyMuPDF

def extract_pages(pdf_path: str) -> list[str]:
    """Extract text from every page of the PDF individually."""
    doc = fitz.open(pdf_path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return pages


raw_pages = extract_pages(PDF_PATH)
print(
    f"Extracted {len(raw_pages)} pages, {sum(len(p) for p in raw_pages):,} characters total"
)

Extracted 3623 pages, 6,277,676 characters total


## 3. Structure the pages by book / chapter


In [3]:
BOOK_TITLES = [
    "Harry Potter and the Sorcerer's Stone",
    "Harry Potter and the Chamber of Secrets",
    "Harry Potter and the Prisoner of Azkaban",
    "Harry Potter and the Goblet of Fire",
    "Harry Potter and the Order of the Phoenix",
    "Harry Potter and the Half-Blood Prince",
    "Harry Potter and the Deathly Hallows",
]

NUMBER_WORDS = [
    "ONE",
    "TWO",
    "THREE",
    "FOUR",
    "FIVE",
    "SIX",
    "SEVEN",
    "EIGHT",
    "NINE",
    "TEN",
    "ELEVEN",
    "TWELVE",
    "THIRTEEN",
    "FOURTEEN",
    "FIFTEEN",
    "SIXTEEN",
    "SEVENTEEN",
    "EIGHTEEN",
    "NINETEEN",
    "TWENTY",
    "TWENTY-ONE",
    "TWENTY-TWO",
    "TWENTY-THREE",
    "TWENTY-FOUR",
    "TWENTY-FIVE",
    "TWENTY-SIX",
    "TWENTY-SEVEN",
    "TWENTY-EIGHT",
    "TWENTY-NINE",
    "THIRTY",
    "THIRTY-ONE",
    "THIRTY-TWO",
    "THIRTY-THREE",
    "THIRTY-FOUR",
    "THIRTY-FIVE",
    "THIRTY-SIX",
    "THIRTY-SEVEN",
    "THIRTY-EIGHT",
    "THIRTY-NINE",
]
NUMBER_VALUE = {word: i + 1 for i, word in enumerate(NUMBER_WORDS)}

# A chapter's first page looks like (drop-cap optional):
#   M
#   CHAPTER ONE
#   THE BOY WHO LIVED
#   r. and Mrs. Dursley ...
CHAPTER_HEADER = re.compile(r"^(?:([A-Z])\n)?\s*CHAPTER\s+([A-Z\-]+)\n([^\n]+)\n\s*")
EPILOGUE_HEADER = re.compile(r"^(?:([A-Z])\n)?\s*EPILOGUE\n([^\n]+)\n\s*")


def fix_title_case(title: str) -> str:
    """Title-case a heading without mangling possessives.
    str.title() turns "Snape's Worst Memory" into "Snape'S Worst Memory" —
    this only capitalizes the first letter of each apostrophe-joined word."""
    return re.sub(
        r"[A-Za-z]+(?:['\u2019][A-Za-z]+)*",
        lambda m: m.group(0)[0].upper() + m.group(0)[1:].lower(),
        title,
    )


def tag_pages_with_structure(raw_pages: list[str]) -> list[dict]:
    """Walk the pages in order, tagging each with its book/chapter.
    A chapter-number reset to 1 marks the start of a new book. TOC pages are dropped."""
    tagged = []
    book_idx = -1
    chapter_num = None
    chapter_title = None

    for i, raw_page in enumerate(raw_pages):
        page_number = i + 1

        if raw_page.strip().upper().startswith("CONTENTS"):
            continue  # table-of-contents page, not book content

        match = CHAPTER_HEADER.match(raw_page)
        if match:
            n = NUMBER_VALUE.get(match.group(2))
            if n is not None:
                if n == 1:
                    book_idx += 1
                chapter_num = n
                chapter_title = fix_title_case(match.group(3).strip())
        else:
            epi_match = EPILOGUE_HEADER.match(raw_page)
            if epi_match:
                chapter_num = "Epilogue"
                chapter_title = fix_title_case(epi_match.group(2).strip())

        if book_idx == -1:
            continue  # front matter before Chapter 1 of Book 1

        tagged.append(
            {
                "page_number": page_number,
                "book_idx": book_idx,
                "book_title": BOOK_TITLES[book_idx],
                "chapter_num": chapter_num,
                "chapter_title": chapter_title,
                "raw_text": raw_page,
            }
        )
    return tagged


tagged_pages = tag_pages_with_structure(raw_pages)
print(f"Tagged {len(tagged_pages)} content pages")
print(f"Books detected: {tagged_pages[-1]['book_idx'] + 1}")

Tagged 3606 content pages
Books detected: 7


## 4. Preprocess / clean each page


In [4]:
def clean_page_text(raw: str) -> str:
    """Clean a single page's raw extracted text."""
    # If the page opens with a chapter/epilogue header, strip the header lines
    # but keep the drop-cap letter (if any), re-joining it onto the paragraph
    # that follows, e.g. "M\nCHAPTER ONE\nTHE BOY WHO LIVED\nr. and Mrs." -> "Mr. and Mrs."
    match = CHAPTER_HEADER.match(raw) or EPILOGUE_HEADER.match(raw)
    if match:
        drop_cap = match.group(1) or ""
        t = drop_cap + raw[match.end() :]
    else:
        t = raw
    t = t.strip()

    # Re-join words split across a line break with a hyphen
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)

    # Collapse newlines (mid-paragraph line wraps) into single spaces
    t = re.sub(r"\n+", " ", t)
    t = re.sub(r"[ \t]+", " ", t)

    # Strip publisher back-matter that trails the very last page
    t = re.split(r"\bISBN \d|\bAll rights reserved\b", t)[0]

    return t.strip()


pages = []
for p in tagged_pages:
    cleaned = clean_page_text(p["raw_text"])
    if len(cleaned) < MIN_PAGE_CHARS:
        continue  # blank / near-empty page (section dividers, stray page breaks)
    if cleaned.startswith("Text copyright") or cleaned.startswith("Cover illustration"):
        continue  # the standalone copyright/credits page at the very end of the book
    pages.append({**{k: v for k, v in p.items() if k != "raw_text"}, "text": cleaned})

print(f"{len(pages)} non-empty pages after cleaning")
print(pages[0]["text"][:400])

3584 non-empty pages after cleaning
Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly normal, thank you very much. They were the last people you’d expect to be involved in anything strange or mysterious, because they just didn’t hold with such nonsense. Mr. Dursley was the director of a firm called Grunnings, which made drills. He was a big, beefy man with hardly any neck, although he did


## 5. Save the structured Markdown file


In [5]:
def pages_to_markdown(pages: list[dict]) -> str:
    lines = []
    current_book, current_chapter = -1, None
    for p in pages:
        if p["book_idx"] != current_book:
            current_book = p["book_idx"]
            lines.append(f"# {p['book_title']}\n")
        if p["chapter_num"] != current_chapter:
            current_chapter = p["chapter_num"]
            label = (
                p["chapter_title"]
                if p["chapter_num"] == "Epilogue"
                else f"Chapter {p['chapter_num']}: {p['chapter_title']}"
            )
            lines.append(f"## {label}\n")
        lines.append(p["text"] + "\n")
    return "\n".join(lines)


markdown_doc = pages_to_markdown(pages)
Path(MD_PATH).write_text(markdown_doc, encoding="utf-8")
print(f"Wrote {MD_PATH} ({len(markdown_doc):,} characters)")

Wrote harry_potter.md (6,271,621 characters)


## 6. Chunk the text — one chunk per page

Each physical page becomes its own chunk, tagged with its page number, book, and chapter. No further splitting or merging across pages.

Field names here (`book_name`, `content`, ...) match the payload schema `rag_api.py` reads at query time — keep them in sync if you change either file.


In [6]:
chunks = []
for p in pages:
    chunks.append(
        {
            "id": len(chunks),
            "page_number": p["page_number"],
            "book_name": p["book_title"],
            "chapter_num": p["chapter_num"],
            "chapter_title": p["chapter_title"],
            "content": p["text"],
        }
    )

print(f"Total chunks (pages): {len(chunks)}")
lengths = [len(c["content"]) for c in chunks]
print(
    f"Avg length: {sum(lengths)/len(lengths):.0f} chars | min: {min(lengths)} | max: {max(lengths)}"
)

Total chunks (pages): 3584
Avg length: 1746 chars | min: 31 | max: 2430


## 7. Embed the chunks

Using `intfloat/multilingual-e5-large` (an E5-family model — the same one `rag_api.py` loads for queries).

E5 models are trained with a `"passage: "` prefix on documents and a `"query: "` prefix on search queries, and expect **normalized** embeddings for cosine similarity. `rag_api.py` already applies the `"query: "` prefix — we mirror that convention here with `"passage: "` so both sides of the pipeline live in the same embedding space.


In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
print(f"Loaded '{EMBEDDING_MODEL}' — embedding dim: {EMBEDDING_DIM}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded 'intfloat/multilingual-e5-large' — embedding dim: 1024


C:\Users\MESK\AppData\Local\Temp\ipykernel_28860\153337555.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()


In [8]:
passages = [f"passage: {c['content']}" for c in chunks]

embeddings = embedding_model.encode(
    passages,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(embeddings.shape)

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

(3584, 1024)


## 8. Store the embeddings locally using Faiss


In [9]:
import faiss
import numpy as np

embeddings_np = np.asarray(embeddings, dtype="float32")

dimension = embeddings_np.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings_np)

print("Vectors:", index.ntotal)
print("Dimension:", dimension)

Vectors: 3584
Dimension: 1024


In [10]:
faiss.write_index(index, "harrypotter.index")

import pickle

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Saved FAISS index and chunks.")

Saved FAISS index and chunks.


## 9. Quick retrieval sanity check


In [11]:
import numpy as np

def retrieve(query: str, top_k: int = 3):
    # E5 models need the "query: " prefix at search time too
    query_vector = embedding_model.encode(f"query: {query}", normalize_embeddings=True)

    query_vector = np.asarray(query_vector, dtype="float32").reshape(1, -1)

    # Search FAISS
    scores, indices = index.search(query_vector, top_k)

    hits = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        # Recreate a simple hit-like object
        chunk = chunks[idx]

        hits.append({"score": float(score), "payload": chunk})

        p = chunk

        chapter_label = (
            p["chapter_title"]
            if p["chapter_num"] == "Epilogue"
            else f"Ch.{p['chapter_num']} {p['chapter_title']}"
        )

        print(
            f"[{score:.3f}] "
            f"{p['book_name']} — "
            f"{chapter_label} "
            f"(p.{p['page_number']})"
        )

        print(f"    {p['content'][:200]}...\n")

    return hits


_ = retrieve("Who is Nicolas Flamel?")

[0.802] Harry Potter and the Sorcerer's Stone — Ch.11 Quidditch (p.175)
    “I’m tellin’ yeh, yer wrong!” said Hagrid hotly. “I don’ know why Harry’s broom acted like that, but Snape wouldn’ try an’ kill a student! Now, listen to me, all three of yeh — yer meddlin’ in things ...

[0.801] Harry Potter and the Sorcerer's Stone — Ch.13 Nicolas Flamel (p.198)
    “Oh, honestly, don’t you two read? Look — read that, there.” She pushed the book toward them, and Harry and Ron read: The ancient study of alchemy is concerned with making the Sorcerer’s Stone, a lege...

[0.789] Harry Potter and the Sorcerer's Stone — Ch.12 The Mirror Of Erised (p.179)
    “I’m sayin’ nothin’,” said Hagrid flatly. “Just have to find out for ourselves, then,” said Ron, and they left Hagrid looking disgruntled and hurried off to the library. They had indeed been searching...



## 10. Generation

The final RAG step: an LLM writes a natural-language answer using the user's question plus the context retrieved from the local FAISS index. rag_api.py uses Gemini for this final generation step in the live API; we do the same here.

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

query = "Who is Nicolas Flamel?"

hits = retrieve(query, top_k=3)

context = "\n\n".join(
    f"Book: {hit['payload']['book_name']}\n"
    f"Page: {hit['payload']['page_number']}\n"
    f"Content: {hit['payload']['content']}"
    for hit in hits
)

gemini_llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL"),
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

messages = [
    SystemMessage(
        content=(
            "Answer only from the provided context. "
            "If the answer is not there, say you do not know. "
            "Keep the answer concise."
        )
    ),
    HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{query}"),
]

response_gemini = gemini_llm.invoke(messages)

print("Answer from Gemini:")
print(response_gemini.content)

[0.802] Harry Potter and the Sorcerer's Stone — Ch.11 Quidditch (p.175)
    “I’m tellin’ yeh, yer wrong!” said Hagrid hotly. “I don’ know why Harry’s broom acted like that, but Snape wouldn’ try an’ kill a student! Now, listen to me, all three of yeh — yer meddlin’ in things ...

[0.801] Harry Potter and the Sorcerer's Stone — Ch.13 Nicolas Flamel (p.198)
    “Oh, honestly, don’t you two read? Look — read that, there.” She pushed the book toward them, and Harry and Ron read: The ancient study of alchemy is concerned with making the Sorcerer’s Stone, a lege...

[0.789] Harry Potter and the Sorcerer's Stone — Ch.12 The Mirror Of Erised (p.179)
    “I’m sayin’ nothin’,” said Hagrid flatly. “Just have to find out for ourselves, then,” said Ron, and they left Hagrid looking disgruntled and hurried off to the library. They had indeed been searching...



c:\Users\MESK\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Answer from Gemini:
[{'type': 'text', 'text': 'Nicolas Flamel is a noted alchemist, opera lover, the owner of the only Sorcerer’s Stone currently in existence, and a friend of Professor Dumbledore. He celebrated his six hundred and sixty-fifth birthday the previous year and enjoys a quiet life in Devon with his wife, Perenelle.', 'extras': {'signature': 'El4KXAERTTIPNXCay01/D1o9hGOHIiKKJqR4d2mojH+t3tSZz6RV9rTtkjvHRSbtWzcThHm3ya0pMAedzGvw4t5FgfryopyJr8R0ETKAbQn2anKReqtiIIZvTauxTO1t'}}]


Same question through Groq, for comparison (this is also the model `rag_api.py` uses for the fast query-routing step).


In [13]:
from groq import Groq

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response_groq = groq_client.chat.completions.create(
    model=os.getenv("GROQ_MODEL"),
    messages=[{"role": "user", "content": query}],
)

print("Answer from Groq:")
print(response_groq.choices[0].message.content)

Answer from Groq:
Nicolas Flamel (c. 1340 – c. 1418) was a French scribe and manuscript dealer who lived in Paris during the late Middle Ages. In the historical record he is known for:

* **Profession:** He ran a successful book‑selling and copying business, which made him relatively wealthy for his time.  
* **Civic life:** He served as a notary and was a respected member of the Parisian guilds; he even held the office of “squire of the king’s household” under Charles VI.  
* **Philanthropy:** Flamel and his wife, Perenelle, funded the construction and renovation of several Parisian churches and charities, most notably the **Church of Saint‑Johann‑des‑Pays** (now known as the **Church of Saint‑Johann de Lurcy**), where a tomb bearing his name still exists.

### The Alchemical Legend

Centuries after his death, Flamel’s name became entwined with legends of alchemy:

1. **The Book of Abraham the Jew:** A 17th‑century text claimed that Flamel possessed a mysterious medieval manuscript th

## 11. Retrieval evaluation: precision and recall

Small test cases using pages we know are relevant, as a simple ground truth.

- **Precision**: of the pages retrieved, how many were actually relevant?
- **Recall**: of the relevant pages, how many did we manage to retrieve?

Add more cases here for a more meaningful evaluation — two is only enough to sanity-check that the pipeline runs end-to-end.


In [14]:
evaluation_cases = [
    {
        "query": "Who rescued Harry from his locked bedroom using a flying car?",
        "relevant_pages": {301, 302},
    },
    {
        "query": "What loophole did Mr Weasley write into the law about enchanting a car?",
        "relevant_pages": {314},
    },
]

TOP_K_EVAL = 3

precision_scores = []
recall_scores = []
retrieved_for_evaluation = []

for case in evaluation_cases:

    query_vector = embedding_model.encode(
        f"query: {case['query']}",
        normalize_embeddings=True
    )

    query_vector = np.asarray(
        query_vector,
        dtype="float32"
    ).reshape(1, -1)

    # FAISS search
    scores, indices = index.search(
        query_vector,
        TOP_K_EVAL
    )

    search_results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        search_results.append({
            "score": float(score),
            "payload": chunks[idx]
        })

    retrieved_pages = {
        result["payload"]["page_number"]
        for result in search_results
    }

    relevant_pages = case["relevant_pages"]

    relevant_retrieved = (
        retrieved_pages & relevant_pages
    )

    precision = (
        len(relevant_retrieved) / len(retrieved_pages)
        if retrieved_pages
        else 0
    )

    recall = (
        len(relevant_retrieved) / len(relevant_pages)
        if relevant_pages
        else 0
    )

    precision_scores.append(precision)
    recall_scores.append(recall)

    retrieved_for_evaluation.append(
        (case, search_results)
    )

    print(case["query"])
    print("Expected pages:", relevant_pages)
    print("Retrieved pages:", retrieved_pages)
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print("-" * 60)


print(
    f"Average precision: "
    f"{sum(precision_scores) / len(precision_scores):.2f}"
)

print(
    f"Average recall: "
    f"{sum(recall_scores) / len(recall_scores):.2f}"
)

Who rescued Harry from his locked bedroom using a flying car?
Expected pages: {301, 302}
Retrieved pages: {3010, 302, 967}
Precision: 0.33
Recall:    0.50
------------------------------------------------------------
What loophole did Mr Weasley write into the law about enchanting a car?
Expected pages: {314}
Retrieved pages: {2056, 314, 467}
Precision: 0.33
Recall:    1.00
------------------------------------------------------------
Average precision: 0.33
Average recall: 0.75


## 12. LLM-as-a-judge evaluation

Gemini answers each test question from the retrieved pages, then a second Gemini call judges that answer for correctness and grounding (i.e. whether it stuck to the provided context instead of inventing details).


In [15]:
## 12. LLM-as-a-judge evaluation

JUDGE_SYSTEM_PROMPT = """You are an evaluator for a question-answering system.
You will be given a context, a question, and an answer.
Judge the answer using only the context.
Return exactly this format:
Score: X/5
Grounded: yes or no
Reason: one short sentence
"""

for case, search_results in retrieved_for_evaluation:

    context = "\n\n".join(
        f"Page {result['payload']['page_number']}: "
        f"{result['payload']['content']}"
        for result in search_results
    )

    # Generate answer
    answer = gemini_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Answer only from the provided context. "
                    "If the answer is not there, say you do not know."
                )
            ),
            HumanMessage(
                content=(
                    f"Context:\n{context}\n\n"
                    f"Question:\n{case['query']}"
                )
            ),
        ]
    ).content

    # Judge answer
    judge = gemini_llm.invoke(
        [
            SystemMessage(
                content=JUDGE_SYSTEM_PROMPT
            ),
            HumanMessage(
                content=(
                    f"Context:\n{context}\n\n"
                    f"Question:\n{case['query']}\n\n"
                    f"Answer:\n{answer}"
                )
            ),
        ]
    ).content

    print("Question:", case["query"])
    print("Answer:", answer)
    print("Judge:", judge)
    print("-" * 60)

c:\Users\MESK\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\MESK\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: Who rescued Harry from his locked bedroom using a flying car?
Answer: [{'type': 'text', 'text': 'Based on the provided context, Ron, Fred, and George came to take Harry, and Fred drove the flying car that pulled the bars out of his window.', 'extras': {'signature': 'El4KXAERTTIP3jGFZoSLm90Q/COdcLHc9EioDqzMkxyUFy1E12cajsozccfzcn8JRcu3OTCuXgLX0vI+cvjzTTBxJrGpoGRwDbY6hK8CUSi5pywDDU7RiU7OUQiB0CPj'}}]
Judge: [{'type': 'text', 'text': 'Score: 5/5\nGrounded: yes\nReason: The answer correctly identifies Ron, Fred, and George as the ones who rescued Harry using a flying car, based on the provided text.', 'extras': {'signature': 'El4KXAERTTIPeQyBGQHPSJ6Vvlkyhf3XWGBYDWEc5/5o0HVt8t9O2/RdcOETxlhS8G9JadQZ9Bz8FQac1VDYlg5qlhN9DbaD9kxSUWpaPLozZfBbx9iYdLHDfxdJlSbX'}}]
------------------------------------------------------------


c:\Users\MESK\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\MESK\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What loophole did Mr Weasley write into the law about enchanting a car?
Answer: [{'type': 'text', 'text': 'Based on the provided context, the loophole Mr. Weasley wrote into the law is that as long as someone was not intending to fly the car, the fact that the car could fly would not be illegal.', 'extras': {'signature': 'El4KXAERTTIPIX1AtZeW51Xz0caFZokeIGipp/27DdBgcboFKDJIjNN7bUxJ3L3m+k45eyw1ByIpfElWUhqvnMSbJWblCLEohxV4duccSvZGsAONMiFEn+XT2rLG7+s8'}}]
Judge: [{'type': 'text', 'text': "Score: 5/5\nGrounded: yes\nReason: The answer accurately reflects the loophole mentioned in the text (that as long as he wasn't intending to fly the car, it wasn't illegal).", 'extras': {'signature': 'El4KXAERTTIP4WHdUS96PyLRX2VaLGtcKqKbqFSKLPyMvXIRRHbaoCXyCcEAhiyONv4duT1ZN2pdZtspn/5t87dQ0QPzd743FMV651nlr1wRi9OQaIgNI6NklWDdTmwF'}}]
------------------------------------------------------------
